In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Omni Flash Video Generation

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fvision%2Fgetting-started%2Fgemini_omni_flash_video_gen.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/vision/getting-started/gemini_omni_flash_video_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Katie Nguyen](https://github.com/katiemn) |

## Overview

Gemini Omni Flash is a model that understands the world around you, allowing you to animate photos or create videos from any input. Built on Gemini's world understanding and native multimodality, Gemini Omni Flash creates outputs that reflect the logic of the real world and lets you shape them step-by-step through natural conversation.

In this tutorial, you'll learn how to use Gemini Omni Flash in Agent Platform with the Google Gen AI SDK to try out the following scenarios:

- Video generation:
  - Text-to-video generation
  - Providing a reference image as a starting frame
  - Supplying reference images to guide video generation
  - Async video generation
- Video editing:
  - Prompt-based video editing
  - Multi-turn video editing (chat)

## Get started

### Install Google Gen AI SDK for Python

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [ ]:
import os
import base64

import matplotlib.image as img
import matplotlib.pyplot as plt
from IPython.display import Video, display
from google import genai
from google.genai import interactions

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)


### Define a helper function to display media

In [ ]:
def show_video(video_bytes):
    data = base64.b64decode(video_bytes)
    with open("sample.mp4", "wb") as out_file:
        out_file.write(data)
    display(Video("sample.mp4", embed=True, width=600))

### Load the video generation model

In [ ]:
omni_model = "gemini-omni-flash-preview"

## Video generation

### Generate videos from a text prompt

With Gemini Omni Flash, you can generate videos directly from text prompts via the [Interactions API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/interactions-api). To generate a video in the sample below, specify the following information:
- **Prompt:** A detailed description of the video you would like to see. For best results, consider attributes such as shot framing, motion, style, lighting, location, and action.
- **Generation Config:** Within this configuration, you can specify a `VideoConfig` that contains a `task` parameter.
  - **Task:** Valid options are `text_to_video`, `image_to_video`, `reference_to_video`, or `edit`. Make sure to set each task accordingly based on your inputs and desired behavior.
- **Video Response Format:** Configure `aspect_ratio`, `duration`, and `delivery` parameters. Alternatively, you can specify these configurations through the text prompt.
  - **Aspect ratio:** 16:9 or 9:16
  - **Duration:** 3s - 10s
  - **Delivery:** If you'd like to save your generated videos to GCS, set `delivery="uri"`. Additionally, set `gcs_uri="gs://<GCS_BUCKET>"`, making sure to replace `<GCS_BUCKET>` with your desired bucket path. To find your video's location, look inside the content list and copy the value next to `uri=` in the `VideoContent` object.


Notes on output videos:
- **Audio generation:** Audio will be generated alongside the output video.
- **Resolution:** 720p

All videos from `gemini-omni-flash-preview` include both [C2PA metadata](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/content-credentials) and a [SynthID watermark](https://deepmind.google/technologies/synthid/).

In [ ]:
prompt = "The words Gemini Omni Flash displayed boldly and centered. The words rapidly and instantly transform, cycling through different styles, fonts, and materials. The words remains perfectly positioned and constant while the colors, textures, and environments instantaneously change around it."

interaction = client.interactions.create(
    model=omni_model,
    input=prompt,
    generation_config=interactions.GenerationConfig(
        video_config=interactions.VideoConfig(
            task="text_to_video"
        )
    ),
    response_format=interactions.VideoResponseFormat(
        aspect_ratio="16:9",
        duration="9s",
        # delivery="uri",
        # gcs_uri="gs://<GCS_BUCKET>"
    )
)

contents = []
for step in interaction.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)

### Video generation from a starting image

#### Download the starting image

You can also generate a video by starting with an input image. In this example, you'll locally download an image that's stored in Google Cloud Storage. If you'd like, you can provide the URL of an image to display it below. If you have a local image you'd like to use, you can specify that in the following steps.

In [ ]:
!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png

If you'd like to use a different local image, modify the file name in `start`.

In [ ]:
start = "suitcase.png"

fig, ax1 = plt.subplots(1, 1, figsize=(12, 6))
ax1.imshow(img.imread(start))
ax1.axis("off")
plt.show()

In [ ]:
prompt = "A single continuous shot of a hard-shell suitcase rolling. It stops, stands vertically, unzips alongside and then opens in half. The wheels and shell stay solid. White bubbly retro text reading 'Omnicase' pops out of the center. Colorful travel stickers pop up around the text: a plane, palm tree, boat, city skyline, and a train."

with open(start, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")

interaction = client.interactions.create(
    model=omni_model,
    input=[
        {"type": "text", "text": prompt},
        {"type": "image", "mime_type": "image/png", "data": img_b64},
    ],
    generation_config=interactions.GenerationConfig(
        video_config=interactions.VideoConfig(
            task="image_to_video"
        )
    ),
)

contents = []
for step in interaction.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)

### Reference inputs in video generation

With Gemini Omni Flash, you can use existing media to generate and edit new videos. The model processes these inputs in one of two ways:

  - **Source Media:** This content acts as the direct foundation for the final output.
    - *Examples:* In image-to-video, the provided image becomes the literal first frame. In video editing, the original video is modified via natural language prompts while keeping its core structure intact.
  - **Reference Media:** This content acts as a guide or inspiration rather than the literal base. You are asking the model to extract a specific style, subject, or mood to creatively reimagine it.
    - *Examples:* Storyboard frames, product images, style guides, audio jingles, or reference scenery. You can also use reference media for video editing—such as providing an image of a new character or art style to apply to a base video.


**NOTE:** Video and audio reference inputs are not currently supported.

#### Download and display reference images

In this example, you'll locally download two images that are stored in Google Cloud Storage.

In [ ]:
!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/woman.jpeg

!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/arcade-game.png

In [ ]:
character = "woman.jpeg"
product = "arcade-game.png"

# Display the images
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(img.imread(character))
ax2.imshow(img.imread(product))
ax1.axis("off")
ax2.axis("off")
plt.show()

**Tip:** You can supply storyboard reference images to better direct the video without supplying all details in a text prompt.

In [ ]:
prompt = "A woman walks up to the Omni-Sphere game on the side wall of a bowling alley. Bass music begins to play as she starts playing the game by pressing a button. 9:16 aspect ratio. 7 second video."

image_files = [character, product]

images_input = []
for img_path in image_files:
    with open(img_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
        images_input.append({"type": "image", "mime_type": "image/jpeg",
  "data": img_b64})

interaction = client.interactions.create(
    model=omni_model,
    input=[
        {"type": "text", "text": prompt},
        *images_input,
    ],
    generation_config=interactions.GenerationConfig(
        video_config=interactions.VideoConfig(
            task="reference_to_video"
        )
    ),
)

contents = []
for step in interaction.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)

### Async video generation

To generate a video that can be checked on later, set the `background` parameter to True. You can then use `interactions.get` to check the status of the initial interaction ID.

In [ ]:
import time

prompt = "Raw cookie ingredients beautifully laid out on a kitchen counter. The view glides smoothly from one ingredient to the next. As each individual ingredient fills the view, a sleek 3D text graphic pops up to label it. The sequence finishes by resting intimately on the rich textures and details of the final ingredient after the label disappears."

initial_interaction = client.interactions.create(
    model=omni_model,
    input=prompt,
    background=True,
)
interaction = initial_interaction

while interaction.status not in ["completed", "failed"]:
    print(f"Current Status: {interaction.status}")
    time.sleep(10)
    interaction = client.interactions.get(id=initial_interaction.id)

if interaction.status == "completed":
    contents = []
    steps = getattr(interaction, "steps", []) or []
    for step in steps:
        if step.type == "model_output":
            contents.extend(step.content)
    show_video(contents[0].data)
else:
    print(f"Status: {interaction.status}")


## Video editing

#### Download and display the video

In this example, you'll locally download a video and an image that are stored in Google Cloud Storage.

**Note:** You can supply reference images when editing source videos. The input source video must be under 10 seconds.

In [ ]:
from PIL import Image as PIL_IMAGE

video_uri = "gs://cloud-samples-data/generative-ai/video/dog_day1.mp4"
image_uri = "gs://cloud-samples-data/generative-ai/image/chair-cat.png"

!gcloud storage cp {video_uri} {"dog_day1.mp4"}
!gcloud storage cp {image_uri} {"chair-cat.png"}

display(Video("dog_day1.mp4", embed=True, width=600))

fig, ax1 = plt.subplots(1, 1, figsize=(6, 6))
ax1.imshow(PIL_IMAGE.open("chair-cat.png"))
ax1.axis("off")
plt.show()

With video editing, you can add, remove, or alter existing objects in the source video. You can also change the content in the video to a different style or reference provided images.

In [ ]:
prompt = "Change the dog to the cat. Remove the backpack and add a propeller hat. Change the tennis balls to balls of yarn."
interaction = client.interactions.create(
    model=omni_model,
    input=[
        {"type": "text", "text": prompt},
        {"type": "image", "mime_type": "image/png", "uri": image_uri},
        {"type": "video", "mime_type": "video/mp4", "uri": video_uri},
    ],
    generation_config=interactions.GenerationConfig(
        video_config=interactions.VideoConfig(
            task="edit"
        )
    ),
)

contents = []
for step in interaction.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)

### Multi-turn editing (chat)

You can iteratively chat with the model to continuously change aspects of the video using the example below.

To successfully accomplish this using the Interactions API, make sure to provide the previous chat history in the `input` parameter.

In [ ]:
prompt1 = "A claymation explainer of Newton's First Law of Motion, everything is made out of clay, no hands, stop motion, sync character's words to mouth movements, show a cute ball character moving and then being stopped by a wall"

interaction1 = client.interactions.create(
    model=omni_model,
    input=prompt1,
)

contents = []
for step in interaction1.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)

In [ ]:
interaction1.steps

In [ ]:
prompt2 = "Now make the same video in a doodle style."

turn2_input = interaction1.steps + [
    {
      "type": "user_input",
      "content": [
        {
          "type": "text",
          "text": prompt2
        }
      ]
    }
]

interaction2 = client.interactions.create(
    model=omni_model,
    input=turn2_input,
)

contents = []
for step in interaction2.steps:
  if step.type == "model_output":
    contents.extend(step.content)

show_video(contents[0].data)



**Note:** Support for audio references, video references, last frame, scene extension, and higher resolutions for the Gemini Omni Flash via Gemini Enterprise Agent Platform API will be available soon.